In [10]:
!pip install chromadb sentence-transformers python-dotenv rich -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.12.0 requires keras<2.13,>=2.12.0, but you have keras 3.10.0 which is incompatible.
tensorflow-intel 2.12.0 requires numpy<1.24,>=1.22, but you have numpy 1.24.3 which is incompatible.
tensorflow-intel 2.12.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 5.29.3 which is incompatible.
tensorflow-intel 2.12.0 requires tensorboard<2.13,>=2.12, but you have tensorboard 2.20.0 which is incompatible.


In [11]:
import re
from pathlib import Path
from typing import List, Dict
import chromadb
import sentence_transformers

print("Imports successful")

Imports successful


In [4]:
CORPUS_DIR = Path("corpus/zoning")

MAX_CHARS = 2400   # ~600 tokens approximation
MIN_CHARS = 50

In [5]:
def load_markdown_files(corpus_dir: Path) -> List[Dict]:

    documents = []

    for file_path in corpus_dir.glob("*.md"):

        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()

        documents.append({
            "source_file": file_path.name,
            "text": text
        })

    return documents

In [6]:
def parse_header(text: str) -> Dict:

    lines = text.split("\n")

    document_title = ""
    last_amended = ""

    for line in lines[:10]:

        line = line.strip()

        # Extract title
        if line.startswith("# "):
            document_title = line.replace("# ", "").strip()

        # Extract amendment date
        if "**Last Amended:**" in line:
            last_amended = (
                line.replace("**Last Amended:**", "")
                .strip()
            )

    return {
        "document_title": document_title,
        "last_amended": last_amended
    }

In [7]:
def split_into_sections(text: str) -> List[Dict]:

    pattern = r"(?=^## )"

    raw_sections = re.split(
        pattern,
        text,
        flags=re.MULTILINE
    )

    sections = []

    for section in raw_sections:

        section = section.strip()

        if not section.startswith("## "):
            continue

        lines = section.split("\n")

        section_title = lines[0].replace("## ", "").strip()

        body = "\n".join(lines[1:]).strip()

        sections.append({
            "section_title": section_title,
            "content": body
        })

    return sections

In [8]:
# def split_large_section(
#     text: str,
#     max_chars: int = MAX_CHARS
# ) -> List[str]:

#     if len(text) <= max_chars:
#         return [text]

#     paragraphs = text.split("\n\n")

#     chunks = []
#     current_chunk = ""

#     for para in paragraphs:

#         para = para.strip()

#         if not para:
#             continue

#         candidate = current_chunk + "\n\n" + para

#         if len(candidate) > max_chars:

#             if current_chunk.strip():
#                 chunks.append(current_chunk.strip())

#             current_chunk = para

#         else:
#             current_chunk = candidate

#     if current_chunk.strip():
#         chunks.append(current_chunk.strip())

#     return chunks

In [26]:
def split_large_section(
    text: str,
    max_chars: int = MAX_CHARS
) -> List[str]:

    if len(text) <= max_chars:
        return [text]

    subsection_pattern = r"(?=^### )"

    subsections = re.split(
        subsection_pattern,
        text,
        flags=re.MULTILINE
    )

    subsections = [
        s.strip()
        for s in subsections
        if s.strip()
    ]

    chunks = []

    current_chunk = ""

    for subsection in subsections:

        candidate = (
            current_chunk
            + "\n\n"
            + subsection
        )

        if len(candidate) > max_chars:

            if current_chunk.strip():
                chunks.append(
                    current_chunk.strip()
                )

            current_chunk = subsection

        else:
            current_chunk = candidate

    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks

In [84]:
def build_chunks(document: Dict) -> List[Dict]:

    source_file = document["source_file"]
    text = document["text"]

    header = parse_header(text)

    sections = split_into_sections(text)

    chunks = []

    chunk_index = 0

    for section in sections:

        section_title = section["section_title"]

        split_chunks = split_large_section(
            section["content"]
        )

        for chunk_text in split_chunks:

            chunk_text = chunk_text.strip()

            if len(chunk_text) < MIN_CHARS:
                continue

            # ---------------------------------
            # Remove redundant markdown metadata
            # ---------------------------------
            
            chunk_text = re.sub(
                r"\*\*Source:\*\*.*?\n",
                "",
                chunk_text
            )
            
            chunk_text = re.sub(
                r"\*\*Last Amended:\*\*.*?\n",
                "",
                chunk_text
            )
            
            chunk_text = re.sub(
                r"\*\*Retrieved:\*\*.*?\n",
                "",
                chunk_text
            )

            prepended_text = (
                f"[Source: {source_file} | "
                f"Section: {section_title} | "
                f"Amended: {header['last_amended']}]\n\n"
                f"{chunk_text}"
            )

            has_cross_ref = bool(
                re.search(
                    r"Section\s+\d{2}-\d{2,3}",
                    chunk_text
                )
            )

            chunks.append({
                "id": f"{source_file}_{chunk_index}",
                "document": prepended_text,
                "metadata": {
                    "source_file": source_file,
                    "section_title": section_title,
                    "last_amended": header["last_amended"],
                    "chunk_index": chunk_index,
                    "has_cross_ref": has_cross_ref
                }
            })

            chunk_index += 1

    return chunks

In [112]:
documents = load_markdown_files(CORPUS_DIR)

all_chunks = []

for document in documents:

    chunks = build_chunks(document)

    all_chunks.extend(chunks)

print(f"Total chunks created: {len(all_chunks)}")

Total chunks created: 17


In [86]:
all_chunks[0]

{'id': 'zr_01_rules_of_construction.md_0',
 'document': '[Source: zr_01_rules_of_construction.md | Section: Section 12-01: Rules Applying to Text of Resolution | Amended: 2/2/2011]\n\n\nThe following rules of construction apply to the text of this Resolution:\n\n(a) The particular shall control the general.\n\n(b) In case of any difference of meaning or implication between the text of this Resolution and any caption, illustration, summary table or illustrative table, the text shall control.\n\n(c) The word "shall" is always mandatory and not discretionary. The word "may" is permissive.\n\n(d) Words used in the present tense shall include the future; and words used in the singular number shall include the plural, and the plural the singular, unless the context clearly indicates the contrary.\n\n(e) A "building" or "structure" includes any part thereof. The terms **residential building**, **commercial building** and **community facility building** shall refer to an entire **building** us

In [87]:
for chunk in all_chunks[:5]:

    print("=" * 80)

    print(chunk["id"])

    print(len(chunk["document"]))

zr_01_rules_of_construction.md_0
2332
zr_01_rules_of_construction.md_1
1148
zr_02_definitions_key.md_0
2265
zr_02_definitions_key.md_1
1593
zr_03_rear_yard_requirements.md_0
1823


In [88]:
import chromadb

from chromadb.utils.embedding_functions import (
    SentenceTransformerEmbeddingFunction
)

In [89]:
CHROMA_PATH = "chroma_db"

COLLECTION_NAME = "zoning_docs"

In [90]:
client = chromadb.PersistentClient(
    path=CHROMA_PATH
)

In [91]:
embedding_function = (
    SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )
)

In [92]:
existing_collections = client.list_collections()

existing_names = [
    collection.name
    for collection in existing_collections
]

if COLLECTION_NAME in existing_names:

    client.delete_collection(
        name=COLLECTION_NAME
    )

    print(f"Deleted existing collection: {COLLECTION_NAME}")

Deleted existing collection: zoning_docs


In [93]:
collection = client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function
)

print("Collection created successfully")

Collection created successfully


In [94]:
documents = [
    chunk["document"]
    for chunk in all_chunks
]

metadatas = [
    chunk["metadata"]
    for chunk in all_chunks
]

ids = [
    chunk["id"]
    for chunk in all_chunks
]

In [95]:
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print("Documents indexed successfully")

Documents indexed successfully


In [96]:
count = collection.count()

print(f"Total indexed documents: {count}")

Total indexed documents: 17


In [97]:
results = collection.query(
    query_texts=[
        "rear yard requirements in R6 districts"
    ],
    n_results=3
)

results

{'ids': [['zr_03_rear_yard_requirements.md_0',
   'zr_03_rear_yard_requirements.md_1',
   'zr_07_front_yard_requirements.md_0']],
 'embeddings': None,
 'documents': [['[Source: zr_03_rear_yard_requirements.md | Section: Section 23-34: Rear Yard and Rear Yard Equivalent Requirements | Amended: 5/12/2021]\n\n\n---\n\n### 23-342: Rear Yard Requirements\n\n**Applicable Districts:** R1 through R10\n\nIn all districts, as indicated, a **rear yard** shall be provided at every required **rear lot line** of a **zoning lot**, except as otherwise provided in Sections 23-341 (Permitted obstructions in required rear yards or rear yard equivalents) and 23-344 (Additional rear yard modifications).\n\nThe minimum required **rear yard** depth shall be:\n\n| District | Minimum Rear Yard Depth |\n|---|---|\n| R1 through R5 | 30 feet |\n| R6 through R10 | 30 feet |\n\nFor corner lots in any Residence District, no rear yard shall be required.\n\nFor through lots in any Residence District, each **street** f

# Cross Reference Map Generation

In [98]:
import json

In [99]:
SECTION_ID_PATTERN = r"(\d{2}-\d{2,3})"

In [100]:
present_sections = set()

for chunk in all_chunks:

    # ---------------------------------
    # Extract from section title
    # ---------------------------------
    section_title = (
        chunk["metadata"]["section_title"]
    )

    title_matches = re.findall(
        SECTION_ID_PATTERN,
        section_title
    )

    for match in title_matches:
        present_sections.add(match)

    # ---------------------------------
    # Extract subsection IDs
    # Example:
    # ### 23-342:
    # ---------------------------------
    text = chunk["document"]

    subsection_matches = re.findall(
        r"###\s+(\d{2}-\d{2,3})",
        text
    )

    for match in subsection_matches:
        present_sections.add(match)

In [101]:
CROSS_REF_PATTERN = r"Section\s+(\d{2}-\d{2,3})"

In [102]:
referenced_sections = set()

for chunk in all_chunks:

    text = chunk["document"]

    matches = re.findall(
        CROSS_REF_PATTERN,
        text
    )

    for match in matches:
        referenced_sections.add(match)

In [103]:
referenced_but_missing = (
    referenced_sections
    - present_sections
)

In [104]:
cross_ref_map = {

    "present": sorted(
        list(present_sections)
    ),

    "referenced_but_missing": sorted(
        list(referenced_but_missing)
    )
}
# cross_ref_map

In [105]:
cross_ref_path = (
    Path(CHROMA_PATH)
    / "cross_ref_map.json"
)

with open(
    cross_ref_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        cross_ref_map,
        f,
        indent=2
    )

print(f"Saved: {cross_ref_path}")

Saved: chroma_db\cross_ref_map.json


In [106]:
print(
    f"Present section IDs: "
    f"{len(present_sections)}"
)

print(
    f"Referenced but missing: "
    f"{len(referenced_but_missing)}"
)

Present section IDs: 17
Referenced but missing: 7


In [107]:
from rich.console import Console
from rich.table import Table

In [108]:
console = Console()

In [109]:
file_stats = []

In [113]:
for document in documents:

    source_file = document["source_file"]

    header = parse_header(
        document["text"]
    )

    chunks = build_chunks(document)

    file_stats.append({

        "source_file": source_file,

        "last_amended": (
            header["last_amended"]
        ),

        "chunks_created": len(chunks)
    })

In [114]:
table = Table(
    title="Zoning Corpus Ingest Summary"
)

table.add_column(
    "Filename",
    style="cyan"
)

table.add_column(
    "Chunks Created",
    justify="right",
    style="green"
)

table.add_column(
    "Last Amended",
    style="yellow"
)

In [115]:
for stat in file_stats:

    table.add_row(

        stat["source_file"],

        str(stat["chunks_created"]),

        stat["last_amended"]
    )

In [116]:
console.print(table)

                         Zoning Corpus Ingest Summary                         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Filename                                   ┃ Chunks Created ┃ Last Amended ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ zr_01_rules_of_construction.md             │              2 │ 2/2/2011     │
│ zr_02_definitions_key.md                   │              2 │ 8/14/2025    │
│ zr_03_rear_yard_requirements.md            │              2 │ 5/12/2021    │
│ zr_04_permitted_obstructions_rear_yard.md  │              1 │ 11/10/2022   │
│ zr_05_floor_area_R6_R12_current.md         │              1 │ 4/30/2024    │
│ zr_06_floor_area_R6_R10_SUPERSEDED_2019.md │              1 │              │
│ zr_07_front_yard_requirements.md           │              1 │ 6/3/2020     │
│ zr_08_permitted_obstructions_all_yards.md  │              1 │ 11/10/2022   │
│ zr_09_ceqr_e_designations.md               │              4 │              │
│ zr_10_height_setback_R6_R12.md             │              2 │ 4/30/2024    │
└────────────────────────────────────────────┴────────────────┴──────────────┘

In [117]:
console.print(
    f"\n[bold green]Total chunks indexed:[/bold green] "
    f"{len(all_chunks)}"
)

Total chunks indexed: 17

In [118]:
console.print(
    f"[bold blue]Present section IDs:[/bold blue] "
    f"{len(present_sections)}"
)

console.print(
    f"[bold red]Referenced but missing:[/bold red] "
    f"{len(referenced_but_missing)}"
)

Present section IDs: 17

Referenced but missing: 7

In [119]:
console.print(
    f"\n[bold yellow]ChromaDB persisted at:[/bold yellow] "
    f"{CHROMA_PATH}"
)

ChromaDB persisted at: chroma_db